# PearlRoute Vision

PearlNet, a custom CNN trained from scratch. All logic is imported from the `pearlroute` package.

## 1. Setup
We import the tools we need and load the three config files (data, model, training).
The real code lives in the `pearlroute` package, so this notebook stays short and just *uses* it.
`SEED` makes random choices (like which sample images we show) the same every run.

In [ ]:
import json
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # hide TensorFlow's low-level info/warning messages

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from pearlroute.config import load_data_config, load_model_config, load_train_config
from pearlroute.data.loaders import list_images, load_class_weights, load_datasets
from pearlroute.preprocessing.stats import compute_channel_stats, load_stats, save_stats
from pearlroute.preprocessing.transforms import read_and_resize

data_cfg, model_cfg, train_cfg = load_data_config(), load_model_config(), load_train_config()
CLASSES = data_cfg.class_names
SEED = data_cfg.seed

# Chart style: light background, quiet grid, no top/right borders.
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#52514e",
    "axes.labelcolor": "#0b0b0b", "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.color": "#e4e3dd", "grid.linewidth": 0.8, "axes.axisbelow": True, "font.size": 10,
})
SPLIT_COLOURS = {"train": "#2a78d6", "val": "#eb6834", "test": "#1baf7a"}
print("classes:", CLASSES)
print("image size:", model_cfg.image_size, "| batch size:", train_cfg.batch_size)

## 2. How many images do we have?
`counts.json` is written by the cleaning and splitting scripts. **raw** is what we downloaded,
**clean** is what survived the quality checks (duplicates, black-and-white photos, tiny images).
The split table shows how the clean images were shared out; demo and retrain images are kept aside.

In [ ]:
counts = json.loads(data_cfg.paths.counts_json.read_text())
cleaning = pd.DataFrame({"raw": counts["cleaning"]["raw"], "clean": counts["cleaning"]["clean"]})
cleaning["dropped"] = cleaning["raw"] - cleaning["clean"]
display(cleaning)

split_table = pd.DataFrame(counts["split"]["per_class"]).T[["train", "val", "test", "retrain", "demo"]]
display(split_table)
print("dropped by reason:", counts["cleaning"]["dropped_by_reason"])

## 3. Chart: images per class in each split
Each bar is one class, cut into train (blue), val (orange) and test (aqua); the number is the total.
All classes should be roughly the same length, and every bar should be about 70 / 15 / 15.
If one class were much shorter, the model would see it less and probably predict it worse.

In [ ]:
order = split_table.sort_values("train").index          # shortest at the bottom
fig, ax = plt.subplots(figsize=(8, 3.6))
left = np.zeros(len(order))
for split, colour in SPLIT_COLOURS.items():
    values = split_table.loc[order, split].to_numpy()
    ax.barh(order, values, left=left, color=colour, height=0.6, label=split,
            edgecolor="#fcfcfb", linewidth=2)               # thin white gap between segments
    left += values
for y, total in enumerate(left):
    ax.text(total + 8, y, f"{int(total)}", va="center", color="#0b0b0b", fontsize=9)
ax.set_xlabel("images")
ax.set_title("Images per class: train / val / test", loc="left", color="#0b0b0b")
ax.grid(axis="y", visible=False)
ax.legend(ncol=3, frameon=False, loc="lower right", bbox_to_anchor=(1, 1.0))
ax.set_xlim(0, left.max() * 1.1)
plt.tight_layout()
plt.show()

## 4. No photographer appears in two splits
One photographer often takes many similar photos of the same place. If some went to train and
some to test, the test would be too easy (the model has already "seen" that scene).
So the split keeps each photographer in one split only; this cell checks that it worked.

In [ ]:
splits = pd.read_csv(data_cfg.paths.splits_csv, dtype=str, keep_default_na=False)
in_splits = splits[splits["split"].isin(["train", "val", "test"])]
leaks = (in_splits.groupby("group")["split"].nunique() > 1).sum()
print("photographers in more than one split:", leaks)
print("photographers per split:", in_splits.groupby("split")["group"].nunique().to_dict())
assert leaks == 0

## 5. Load the images with tf.data
`load_datasets` reads every image, resizes it to a square, and groups the images into batches.
Only the training set is shuffled, so the model doesn't see the classes in a fixed order.
A batch is a 4-D array: (images in batch, height, width, 3 colour channels). Labels are class numbers.

In [ ]:
raw_sets = load_datasets(data_cfg, model_cfg, train_cfg)          # pixels still 0-255
images, labels = next(iter(raw_sets["train"]))
print("batch shape:", images.shape, "| dtype:", images.dtype)
print("pixel range:", float(tf.reduce_min(images)), "to", float(tf.reduce_max(images)))
print("first labels:", labels[:8].numpy(), "->", [CLASSES[i] for i in labels[:4].numpy()])
print("class weights:", load_class_weights(data_cfg), "(None = classes are balanced enough)")

## 6. What the model actually sees
These are random training images after resizing to the model's input size, one row per class.
Photos are squashed to a square rather than cropped, so nothing is cut off; a little stretching is normal.
This is a quick sanity check that every class folder holds the kind of photo its name promises.

In [ ]:
paths, path_labels = list_images(data_cfg.paths.processed_dir / "train", CLASSES)
paths, path_labels = np.array(paths), np.array(path_labels)
rng = np.random.default_rng(SEED)
per_row = 6
fig, axes = plt.subplots(len(CLASSES), per_row, figsize=(per_row * 1.6, len(CLASSES) * 1.75))
for row, name in enumerate(CLASSES):
    picks = rng.choice(paths[path_labels == row], per_row, replace=False)
    for col, path in enumerate(picks):
        ax = axes[row, col]
        ax.imshow(read_and_resize(tf.constant(path), model_cfg.image_size).numpy())
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        for spine in ax.spines.values():
            spine.set_visible(False)
    axes[row, 0].set_ylabel(name.replace("_", "\n"), rotation=0, ha="right", va="center", fontsize=9)
fig.suptitle(f"Random training images at {model_cfg.image_size} x {model_cfg.image_size}", x=0.02, ha="left")
plt.tight_layout()
plt.show()

## 7. Normalisation: work out the average colour of the training set
Normalising means: for each colour channel, subtract the average and divide by the spread (std).
Afterwards pixel values sit around 0 instead of 0-255, which helps the network train faster and more stably.
We use **training images only**: peeking at val/test would leak information about them.
The numbers are saved to `data/stats.json` so the app normalises uploaded photos exactly the same way.

In [ ]:
mean, std, n = compute_channel_stats(raw_sets["train"])
save_stats(data_cfg.paths.stats_json, mean, std, model_cfg.image_size, n)
print(f"computed on {n} training images")
print("mean (R, G, B):", mean)
print("std  (R, G, B):", std)

## 8. Chart: pixel values before and after normalisation
Left: how often each pixel value (0-255) appears in one batch, for red, green and blue.
Right: the same pixels after normalising, now centred on 0 with most values between about -2 and 2.
The shape doesn't change, only the scale moves; that's exactly what normalising should do.

In [ ]:
stats = load_stats(data_cfg.paths.stats_json)
norm_sets = load_datasets(data_cfg, model_cfg, train_cfg, stats=stats)
before, _ = next(iter(raw_sets["val"]))
after, _ = next(iter(norm_sets["val"]))            # same images: val is not shuffled

channels = [("R", "#e34948", "-"), ("G", "#008300", "--"), ("B", "#2a78d6", ":")]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.4))
for ax, batch, bins, title in [(ax1, before, np.linspace(0, 255, 52), "Before: pixel values 0-255"),
                               (ax2, after, np.linspace(-3, 3, 52), "After: normalised")]:
    for c, (name, colour, style) in enumerate(channels):
        values = batch.numpy()[..., c].ravel()
        hist, edges = np.histogram(values, bins=bins, density=True)
        centres = (edges[:-1] + edges[1:]) / 2
        ax.plot(centres, hist, color=colour, linestyle=style, linewidth=2)
        peak = hist.argmax()
        ax.annotate(name, (centres[peak], hist[peak]), xytext=(4, 4), textcoords="offset points",
                    color="#0b0b0b", fontsize=9, fontweight="bold")
    ax.set_title(title, loc="left", color="#0b0b0b")
    ax.set_xlabel("pixel value")
    ax.set_yticks([])
ax1.set_ylabel("how common")
ax2.axvline(0, color="#52514e", linewidth=1)
fig.legend([plt.Line2D([], [], color=c, linestyle=s, linewidth=2) for _, c, s in channels],
           [n for n, _, _ in channels], ncol=3, frameon=False, loc="upper right")
plt.tight_layout()
plt.show()

## 9. Check: are the normalised values centred?
On the training set the mean should be almost exactly 0 and the std almost exactly 1, by construction.
Validation images were not used to compute the stats, so they land close to 0 and 1 but not exactly.
Close is good: it means val photos look like train photos, colour-wise.

In [ ]:
def overall_mean_std(dataset):
    m, s, _ = compute_channel_stats(dataset)
    return np.round(m, 3), np.round(s, 3)

for split in ["train", "val"]:
    m, s = overall_mean_std(norm_sets[split])
    print(f"{split:5s} mean {m}  std {s}")

## 10. Augmentation: small random changes to training photos
Each epoch, every training photo gets randomly flipped left-right, tilted a little, zoomed, shifted,
and made brighter/darker. The model never sees the exact same picture twice, so it has to learn
the subject (a raft, a gorilla) instead of memorising pixels. We skip upside-down flips and colour
shifts, because skies are up and colour matters here (green forest, white water).

In [ ]:
from pearlroute.preprocessing.augmentation import make_augment

print("settings:", train_cfg.augmentation.model_dump())
augment = make_augment(train_cfg.augmentation, seed=SEED)

## 11. Chart: one photo, many versions
The left column is the original photo; the other columns are what augmentation might produce from it.
Every version should still clearly show the same experience. If a change made the photo
unrecognisable (e.g. the raft cut off), the settings in `configs/train.yaml` would be too strong.

In [ ]:
# Pick one training photo per class whose title names the subject, so the example is a clear one.
train_rows = splits[splits["split"] == "train"]
rng = np.random.default_rng(SEED)
examples = []
for name, word in [("whitewater_rafting", "raft"), ("gorilla_primate_trekking", "gorilla"), ("waterfalls", "falls")]:
    rows = train_rows[(train_rows["label"] == name) & train_rows["title"].str.contains(word, case=False)]
    examples.append(str(data_cfg.paths.raw_dir / rng.choice(rows["filename"].to_numpy())))
versions = 6
fig, axes = plt.subplots(len(examples), versions, figsize=(versions * 1.7, len(examples) * 1.8))
for row, path in enumerate(examples):
    original = tf.cast(read_and_resize(tf.constant(path), model_cfg.image_size), tf.float32)
    batch = tf.repeat(original[None], versions - 1, axis=0)        # 5 copies of the same photo
    shown = [original] + list(augment(batch))
    for col, img in enumerate(shown):
        ax = axes[row, col]
        ax.imshow(np.asarray(img).astype("uint8"))
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        for spine in ax.spines.values():
            spine.set_visible(False)
    axes[row, 0].set_title("original", fontsize=9, loc="left")
    axes[row, 1].set_title("augmented ->", fontsize=9, loc="left")
fig.suptitle("Augmentation: same photo, different random changes", x=0.02, ha="left")
plt.tight_layout()
plt.show()

## 12. The full training pipeline
This is exactly what the model will be fed: resize -> augment (train only) -> normalise.
Augmentation runs while the images are loaded, so it costs a little time each epoch but no disk space.
Validation and test images are never augmented: we want to score the model on real, unchanged photos.

In [ ]:
import time

train_sets = load_datasets(data_cfg, model_cfg, train_cfg, augment=augment, stats=stats)
start = time.time()
batches = sum(1 for _ in train_sets["train"])
print(f"one training epoch of images: {batches} batches in {time.time() - start:.1f}s")
x, y = next(iter(train_sets["train"]))
print("train batch:", x.shape, "| mean", round(float(tf.reduce_mean(x)), 2), "| std", round(float(tf.math.reduce_std(x)), 2))
x_val, _ = next(iter(train_sets["val"]))
x_val_plain, _ = next(iter(norm_sets["val"]))
print("val batch unchanged by augmentation:", bool(np.allclose(x_val.numpy(), x_val_plain.numpy())))

## 13. Model A: the baseline CNN
A plain CNN: 4 blocks of *convolution* (small filters that detect edges, then textures, then shapes)
followed by *max-pooling* (shrinks the image by half, keeping the strongest signals).
Then one dense layer combines everything, dropout randomly switches off half its units to reduce
overfitting, and softmax turns the scores into 6 probabilities that add up to 1. This is the score to beat.

In [ ]:
from pearlroute.models.registry import build_model

baseline = build_model("baseline", model_cfg, len(CLASSES))
baseline.summary()

## 14. Train the baseline (or load the saved one)
Training shows the model every training photo once per *epoch* and nudges its weights to reduce the loss.
Early stopping ends training when validation loss stops improving and keeps the best epoch's weights.
It takes ~30 minutes on a CPU, so by default we load the model saved by `scripts/train.py`;
set `RETRAIN_BASELINE = True` to train it again here.

In [ ]:
from pearlroute.models.io import load_model
from pearlroute.training.run import train_and_evaluate
from pearlroute.training.trainer import load_history

RETRAIN_BASELINE = False
baseline_dir = train_cfg.models_dir / "baseline"

if RETRAIN_BASELINE or not (baseline_dir / "model.keras").exists():
    result = train_and_evaluate("baseline", data_cfg, model_cfg, train_cfg)
    baseline = result["model"]
else:
    baseline, baseline_meta = load_model(baseline_dir)
baseline_history = load_history(baseline_dir)
baseline_metrics = json.loads((baseline_dir / "metrics.json").read_text())
print(f"trained {baseline_metrics['epochs_run']} epochs in {baseline_metrics['train_seconds'] / 60:.0f} min;"
      f" best epoch: {baseline_metrics['best_epoch']}")

## 15. Chart: learning curves
Blue is the training set, orange (dashed) is validation; the dotted line marks the best epoch.
If both curves improve together, the model is learning. If training keeps improving while validation
gets worse, the model is *overfitting*: memorising training photos instead of learning general patterns.

In [ ]:
from pearlroute.evaluation.plots import plot_confusion_matrix, plot_history

plot_history(baseline_history, "Baseline: learning curves")
plt.show()

## 16. How good is the baseline? (test set)
The test set was never used for training or choosing the best epoch, so this is the fair score.
**Accuracy** = share of photos classified correctly. **Precision** = when it says "waterfalls", how often it's right;
**recall** = of all waterfall photos, how many it found; **F1** combines both. **ROC-AUC** (1.0 = perfect,
0.5 = guessing) checks how well the probabilities rank the right class above the wrong ones.

In [ ]:
test_m = baseline_metrics["test"]
display(pd.DataFrame({name: [test_m[key]] for name, key in [
    ("accuracy", "accuracy"), ("precision (macro)", "precision_macro"), ("recall (macro)", "recall_macro"),
    ("F1 (macro)", "f1_macro"), ("ROC-AUC (macro)", "roc_auc_macro"), ("loss", "loss")]}, index=["baseline"]))
display(pd.DataFrame(test_m["per_class"]).T.sort_values("f1-score"))

## 17. Chart: confusion matrix
Each row is the true class, each column what the model predicted; the diagonal is correct answers.
Dark squares off the diagonal show which experiences the model mixes up, e.g. waterfalls vs rafting
(both white water) or mountains vs gorilla trekking (both green forest).

In [ ]:
plot_confusion_matrix(test_m["confusion_matrix"], CLASSES, "Baseline: test set confusion matrix")
plt.show()

## 18. Look at some mistakes
Numbers don't tell you *why* the model is wrong, pictures do. These are test photos the baseline got wrong,
with the true class, the prediction and how confident it was. Some mistakes are the model's fault;
some are photos that were labelled wrongly when we collected them.

In [ ]:
from pearlroute.evaluation.metrics import predict_dataset

test_paths, _ = list_images(data_cfg.paths.processed_dir / "test", CLASSES)
y_true, y_prob = predict_dataset(baseline, norm_sets["test"])          # same order as test_paths
y_pred = y_prob.argmax(axis=1)
wrong = np.flatnonzero(y_pred != y_true)
picks = np.random.default_rng(SEED).choice(wrong, min(12, len(wrong)), replace=False)
fig, axes = plt.subplots(3, 4, figsize=(10, 9.6))
for ax, i in zip(axes.ravel(), picks):
    ax.imshow(read_and_resize(tf.constant(test_paths[i]), model_cfg.image_size).numpy())
    ax.set_title(f"true: {CLASSES[y_true[i]]}\npred: {CLASSES[y_pred[i]]} ({y_prob[i].max():.0%})", fontsize=7.5, loc="left")
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
fig.suptitle(f"Baseline: {len(wrong)} of {len(y_true)} test photos wrong; 12 examples", x=0.01, ha="left")
plt.tight_layout()
plt.show()

## 19. Model B: PearlNet
PearlNet fixes the baseline's weak spots. **Batch norm** after each convolution keeps training steady.
**Residual blocks** add a shortcut around each pair of convolutions, so a deeper network still trains well.
**Squeeze-excitation (SE)** lets each block turn useful channels up and others down for each photo.
**Global average pooling** replaces the baseline's huge Flatten + Dense layer (1.6 M weights in one layer).

In [ ]:
pearlnet = build_model("pearlnet", model_cfg, len(CLASSES))
print(f"PearlNet: {pearlnet.count_params():,} parameters in {len(pearlnet.layers)} layers "
      f"(baseline: {baseline.count_params():,})")
pearlnet.summary(line_length=100, expand_nested=False, show_trainable=False)

## 20. Train PearlNet (or load the saved one)
Same data, same augmentation, same training settings as the baseline, so the comparison is fair.
PearlNet is deeper, so each epoch takes about 3x longer (~100 s on a CPU, roughly an hour in total).
By default we load the model saved by `scripts/train.py --model pearlnet`; set `RETRAIN_PEARLNET = True` to retrain here.

In [ ]:
RETRAIN_PEARLNET = False
pearlnet_dir = train_cfg.models_dir / "pearlnet"

if RETRAIN_PEARLNET or not (pearlnet_dir / "model.keras").exists():
    pearlnet = train_and_evaluate("pearlnet", data_cfg, model_cfg, train_cfg)["model"]
else:
    pearlnet, pearlnet_meta = load_model(pearlnet_dir)
pearlnet_history = load_history(pearlnet_dir)
pearlnet_metrics = json.loads((pearlnet_dir / "metrics.json").read_text())
print(f"trained {pearlnet_metrics['epochs_run']} epochs in {pearlnet_metrics['train_seconds'] / 60:.0f} min;"
      f" best epoch: {pearlnet_metrics['best_epoch']}")

## 21. Chart: PearlNet learning curves
Compare this with section 15. Validation accuracy climbs higher than the baseline's (~0.64 vs ~0.61),
but training accuracy climbs even faster (~0.9), so the gap between the lines is *wider*: PearlNet
learns more, and also memorises more. More regularisation (or cleaner labels) is the obvious next lever.

In [ ]:
plot_history(pearlnet_history, "PearlNet: learning curves")
plt.show()

## 22. Baseline vs PearlNet on the test set
Both models are scored on the same 591 test photos that neither saw during training.
Higher is better for every column except loss. "Train-val gap" is training accuracy minus validation
accuracy at the best epoch: a big gap means overfitting.

In [ ]:
def summary_row(metrics, history):
    t = metrics["test"]
    best = history.loc[metrics["best_epoch"]]
    return {"test accuracy": t["accuracy"], "F1 (macro)": t["f1_macro"], "precision (macro)": t["precision_macro"],
            "recall (macro)": t["recall_macro"], "ROC-AUC (macro)": t["roc_auc_macro"], "test loss": t["loss"],
            "train-val gap": round(best["accuracy"] - best["val_accuracy"], 3),
            "parameters": metrics["parameters"], "train minutes": round(metrics["train_seconds"] / 60, 1)}

comparison = pd.DataFrame({"baseline": summary_row(baseline_metrics, baseline_history),
                           "pearlnet": summary_row(pearlnet_metrics, pearlnet_history)}).T
display(comparison)

## 23. Chart: F1 score per class, baseline vs PearlNet
Each pair of bars is one class: blue = baseline, orange = PearlNet, longer is better.
This shows *where* PearlNet improved (or didn't), which the single accuracy number hides.

In [ ]:
f1 = pd.DataFrame({name: {c: m["test"]["per_class"][c]["f1-score"] for c in CLASSES}
                   for name, m in [("baseline", baseline_metrics), ("pearlnet", pearlnet_metrics)]})
f1 = f1.sort_values("pearlnet")
fig, ax = plt.subplots(figsize=(8, 4.4))
y = np.arange(len(f1))
for offset, (name, colour) in zip([0.2, -0.2], [("baseline", "#2a78d6"), ("pearlnet", "#eb6834")]):
    ax.barh(y + offset, f1[name], height=0.38, color=colour, label=name, edgecolor="#fcfcfb", linewidth=2)
    for yi, v in zip(y + offset, f1[name]):
        ax.text(v + 0.01, yi, f"{v:.2f}", va="center", fontsize=8, color="#0b0b0b")
ax.set_yticks(y, [c.replace("_", " ") for c in f1.index])
ax.set_xlim(0, 1)
ax.set_xlabel("F1 score on the test set")
ax.grid(axis="y", visible=False)
ax.set_title("F1 per class: baseline vs PearlNet", loc="left", color="#0b0b0b")
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.show()

## 24. Chart: PearlNet confusion matrix
Same reading as section 17: the diagonal is correct, off-diagonal squares are mix-ups.
Compare it with the baseline's matrix to see which confusions PearlNet fixed and which remain.

In [ ]:
plot_confusion_matrix(pearlnet_metrics["test"]["confusion_matrix"], CLASSES, "PearlNet: test set confusion matrix")
plt.show()

## 25. Which model do we keep?
We pick the model with the higher **macro F1** on the test set, because it treats every class equally,
including the smaller mountain_hiking class. The chosen model is the one the app will use for predictions.
With 591 test photos, a 5-point gain is a real but modest improvement, not a breakthrough.

In [ ]:
winner = comparison["F1 (macro)"].idxmax()
loser = comparison.index.drop(winner)[0]
print(f"Winner: {winner} (macro F1 {comparison.loc[winner, 'F1 (macro)']:.3f} vs "
      f"{comparison.loc[loser, 'F1 (macro)']:.3f} for {loser}; "
      f"accuracy {comparison.loc[winner, 'test accuracy']:.1%} vs {comparison.loc[loser, 'test accuracy']:.1%})")

## Next steps
- Save the winning model as the app's active model and build the predictor.
- API (upload, predict, retrain) and the Streamlit UI.